all required installs:

pip install imbalanced-learn
pip install geopandas
pip install pandas
pip install matplotlib
pip install scikit-learn

In [13]:
# import 

import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


In [ ]:
# CSV Stuff

habitats_df = pd.read_csv('data/habitats_cbs_2022.csv')

df = pd.read_csv(
    'data/Streptopelia turtur.csv/Streptopelia turtur.csv',
    dtype={'Streptopelia turtur': 'object'}
)

In [20]:
df['Streptopelia turtur'] = pd.to_numeric(df['Streptopelia turtur'], errors='coerce')
df['Streptopelia turtur'] = df['Streptopelia turtur'].fillna(0)
df['Streptopelia turtur'] = df['Streptopelia turtur'].apply(lambda x: 1 if x > 0 else 0)

# Convert and extract date parts from 'eventDate'
date_col = 'eventDate' 
df[date_col] = pd.to_datetime(df[date_col])
df['year'] = df[date_col].dt.year
df['month'] = df[date_col].dt.month
df['day'] = df[date_col].dt.day
df['day_of_week'] = df[date_col].dt.dayofweek # 0 = Monday, 6 = Sunday

# Drop the original text date column so the model doesn't see it
df = df.drop(columns=[date_col])

X = df.drop(columns=['Streptopelia turtur'])
y = df['Streptopelia turtur']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rus = RandomUnderSampler(random_state=42)
X_train_resampled, y_train_resampled = rus.fit_resample(X_train, y_train)

df.to_csv('data/downSampBirdData.csv', index=False)

In [21]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_resampled, y_train_resampled)

y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9163048350995597
[[2298792  209952]
 [    270    2744]]
              precision    recall  f1-score   support

           0       1.00      0.92      0.96   2508744
           1       0.01      0.91      0.03      3014

    accuracy                           0.92   2511758
   macro avg       0.51      0.91      0.49   2511758
weighted avg       1.00      0.92      0.96   2511758

